# Lektion 12: Der Graph füllt sich selbst – LLM-Extraktion & NL→Cypher



| | |
|---|---|
| **Zeit** | ~75 Minuten |
| **Schwierigkeit** | Fortgeschritten |
| **Voraussetzungen** | Lektion 10–11 |
| **Du baust** | Eine Pipeline: Text → LLM-Extraktion → Graph (mit Entity Resolution) → Fragen auf Deutsch → Cypher → Antwort |

## 🎯 Nach dieser Lektion kannst du:

- ein LLM Entitäten und Beziehungen **strukturiert** extrahieren lassen (Pydantic-Vertrag)

- **Entity Resolution** implementieren (keine Duplikate)

- natürliche Fragen in Cypher übersetzen – abgesichert durch die **Whitelist**

- einen Angriffs-Test durchführen (`DETACH DELETE` wird abgelehnt)

- die zwei Risiken (Injection, Doppel-Entities) und ihre Bremsen benennen

## ⚙️ Setup – einmal pro Notebook ausführen

Diese Zelle findet die Projektwurzel und lädt die **zentrale KI-Konfiguration** aus deiner `.env` (siehe `docs/local-ai-server.md`). Du musst nie Notebook-Code ändern, um den Anbieter zu wechseln.

In [1]:
import sys
from pathlib import Path

# Projektwurzel finden (dort liegt der Ordner "lernakademie")
ROOT = Path.cwd()
while not (ROOT / "lernakademie").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "lernakademie").exists():
    raise RuntimeError("Projektwurzel nicht gefunden – starte das Notebook im Repo-Ordner.")
sys.path.insert(0, str(ROOT))

from lernakademie.config import llm_zusammenfassung

_info = llm_zusammenfassung()  # zeigt NIE den API-Key selbst
print("KI-Konfiguration:", _info["anbieter"], "| Modell:", _info["modell"],
      "| Key:", _info["api_key"])
if _info["anbieter"] == "offline":
    print("Offline-Modus: alles laeuft – Antworten sind Simulationen ([OFFLINE]).")


KI-Konfiguration: offline | Modell: offline-simulation | Key: nicht gesetzt (optional)
Offline-Modus: alles laeuft – Antworten sind Simulationen ([OFFLINE]).


## 🧠 Konzept: Extraktion – der Leser mit Formular

**WARUM:** Bislang hast du den Graph von Hand gepflegt. Echte Systeme lassen ihn aus Dokumenten **wachsen**: Text rein → Entitäten + Beziehungen raus → Graph gefüllt.

**WAS:** Das LLM hält statt einer freien Zusammenfassung ein **Formular** (Pydantic, Lektion 09!): `entitaeten` (Name + Label) und `beziehungen` (von, Typ, zu). **Das LLM schlägt vor – Python entscheidet.**

In [2]:
from lernakademie.graph_extraction import extrahiere

TEXT = ("Anna arbeitet bei Acme und gehoert zu Engineering. "
        "Ben arbeitet bei Acme. Engineering arbeitet an AI Assistant.")

extraktion = extrahiere(TEXT)
print("Entitaeten:")
for e in extraktion.entitaeten:
    print("  -", e["name"], f"({e['label']})")
print("Beziehungen:")
for b in extraktion.beziehungen:
    print(f"  - {b['von']} --{b['typ']}--> {b['zu']}")

Entitaeten:
  - Anna (Person)
  - Ben (Person)
  - Acme (Company)
  - Engineering (Department)
  - AI Assistant (Project)
Beziehungen:
  - Anna --WORKS_FOR--> Acme
  - Anna --BELONGS_TO--> Engineering
  - Engineering --WORKS_ON--> AI Assistant


### 🔎 Was sehen wir hier?
Die Extraktion ist ein **geprüftes Objekt** (kein freier Text!). Offline nutzt die Akademie eine Regel-Extraktion (bekannte Namen + Muster „X arbeitet bei Y") – mit echtem Modell extrahiert Mistral frei. Die Schnittstelle bleibt identisch.

**TRY:** Führe die Extraktion mit einem Tippfehler („Anya arbeitet bei Acme“) aus – was passiert ehrlich?

## 🧠 Konzept: Entity Resolution – der Türsteher

**Metapher:** Vor dem Club steht ein Türsteher mit Gästeliste: "Anna? Steht schon drauf – kein zweiter Stempel!" Vor jedem `add_node` steht deshalb ein `find_node`-Check. **Das verhindert Graph-Müll** – denn doppelt gespeicherte Entities zerreißen jede Traversierung.

In [3]:
from lernakademie.graph_extraction import fuelle_graph
from lernakademie.graph_store import GraphStore

store = GraphStore()

kanten1 = fuelle_graph(store, extrahiere(TEXT))
print(f"Erster Lauf: {len(store.nodes())} Nodes, {kanten1} Kanten")

kanten2 = fuelle_graph(store, extrahiere(TEXT))   # DER DUPLIKAT-TEST
print(f"Zweiter Lauf: {len(store.nodes())} Nodes, {kanten2} Kanten")
print("-> Nodes duerfen NICHT gestiegen sein (Entity Resolution wirkt).")

Erster Lauf: 5 Nodes, 3 Kanten
Zweiter Lauf: 5 Nodes, 3 Kanten
-> Nodes duerfen NICHT gestiegen sein (Entity Resolution wirkt).


## ✏️ Deine Aufgabe: Der Duplikat-Beweis

Du implementierst die Entity-Resolution selbst und beweist ihre Wirkung.

**Anforderungen:**

- Schreibe `fuelle_graph_eigen(store, extraktion)`, das alle Entitäten und Beziehungen anlegt.

- **Vor jedem Anlegen:** prüfe mit `find_node`, ob die Entity schon existiert (Existing-ID wiederverwenden!).

- Führe die Funktion zweimal mit derselben Extraktion aus und vergleiche die Node-Anzahl.

**Schreibe deinen Code in die folgende Zelle:**

In [4]:
def fuelle_graph_eigen(store, extraktion):
    ids = {}
    for entitaet in extraktion.entitaeten:
        name, label = entitaet["name"], entitaet.get("label", "Entity")
        vorhanden = store.find_node(label, name=name)
        ids[name] = vorhanden.id if vorhanden else store.add_node([label], name=name)
    gesetzt = 0
    for beziehung in extraktion.beziehungen:
        von, zu = beziehung["von"], beziehung["zu"]
        if von in ids and zu in ids:      # unvollstaendig -> ueberspringen
            store.add_relationship(ids[von], ids[zu], beziehung["typ"])
            gesetzt += 1
    return gesetzt

fuelle_graph_eigen(store, extrahiere(TEXT))
vorher = len(store.nodes())
fuelle_graph_eigen(store, extrahiere(TEXT))
print("Vorher:", vorher, "| Nachher:", len(store.nodes()), "(beweis: identisch)")

Vorher: 5 | Nachher: 5 (beweis: identisch)


**💡 Warum diese Loesung funktioniert:**

Der `find_node`-Check pro Entity ist der Türsteher. Unvollständige Beziehungen überspringst du bewusst – raten wäre schlimmer als weglassen. Du hast gerade die wichtigste Qualitätsregel jedes Knowledge Graphs implementiert.

### ✅ Richtig, wenn:

- [ ] Der zweite Lauf erzeugt **null** neue Nodes.

- [ ] Unvollständige Beziehungen (unbekannte Entity) werden übersprungen – kein Absturz.

- [ ] Kanten verbinden IDs, niemals Namen.

**🧪 Fuehre die folgende Zelle aus, um dein Ergebnis zu pruefen:**

In [5]:
neu = GraphStore()
ex = extrahiere(TEXT)
fuelle_graph_eigen(neu, ex)
stand = len(neu.nodes())
fuelle_graph_eigen(neu, ex)
assert len(neu.nodes()) == stand, "Entity Resolution: Nodes duerfen beim 2. Lauf nicht steigen!"
assert stand >= 3, "Mindestens Anna, Acme, Engineering sollten angelegt sein."
print("Alle Prüfungen bestanden! ✅")

Alle Prüfungen bestanden! ✅


<details><summary>💡 Hinweis 1 (oeffnen, wenn du nicht weiterkommst)</summary>

`vorhanden = store.find_node(label, name=name)` – `None` heißt: neu anlegen.
</details>

<details><summary>💡 Hinweis 2 (oeffnen, wenn du nicht weiterkommst)</summary>

IDs im `ids`-Dictionary sammeln: `ids[name] = node.id`.
</details>

<details><summary>💡 Hinweis 3 (oeffnen, wenn du nicht weiterkommst)</summary>

Für Beziehungen: `if von in ids and zu in ids: store.add_relationship(...)` – sonst überspringen.
</details>

## 📝 Reflexion

Beantworte in deinen eigenen Worten:

1. Warum führt "das LLM darf Labels frei erfinden" zu Graph-Chaos – und wie wirkt die Struktur dagegen?

2. Was kostet die Extraktion bei 500 Dokumenten – und wo würdest du cachen?

## 🧠 Konzept: Natürliche Sprache → Cypher (mit Bremse!)

**WARUM:** Nutzer fragen in Worten („Wo arbeitet Anna?"), der Graph versteht Cypher. Die Übersetzung übernimmt ein LLM (offline: Regel-Fallback) – **ABER:**

> **Generierte Abfragen sind nicht vertrauenswürdig.** Ein manipulierter Prompt kann `DELETE` erzeugen. Deshalb: deterministische **Whitelist vor** jeder Ausführung. Die Prüfung muss Python-Code sein – **niemals ein LLM**, denn LLMs prüfen Sicherheitsregeln nicht zuverlässig.

In [6]:
from lernakademie.graph_extraction import cypher_aus_frage, ist_sicher, fuehre_sicher_aus

fragen = ["Wo arbeitet Anna?", "Was gibt es ueber Acme?", "Wie hoch ist das Weihnachtsgeld?"]
for frage in fragen:
    cypher = cypher_aus_frage(frage)
    print(f"FRAGE: {frage}")
    print(f"  Cypher: {cypher}")
    if cypher and ist_sicher(cypher):
        ergebnis = fuehre_sicher_aus(store, cypher)
        print(f"  -> {len(ergebnis)} Treffer")
    else:
        print("  -> ehrlich: nicht uebersetzbar / keine Treffer")
    print()

FRAGE: Wo arbeitet Anna?
  Cypher: MATCH (a {name: 'Anna'})-[r]->(b) RETURN a, type(r), b
  -> 8 Treffer

FRAGE: Was gibt es ueber Acme?
  Cypher: MATCH (n:Company {name: 'Acme'}) RETURN n
  -> 1 Treffer

FRAGE: Wie hoch ist das Weihnachtsgeld?
  Cypher: None
  -> ehrlich: nicht uebersetzbar / keine Treffer



In [7]:
# DER ANGRIFFS-TEST (Pflicht in dieser Lektion!):
boese = "MATCH (n:Person) DETACH DELETE n"
try:
    fuehre_sicher_aus(store, boese)
    print("FEHLER IM DESIGN: destruktive Abfrage wurde ausgefuehrt!")
except ValueError as fehler:
    print("Korrekt abgelehnt ✅")
    print("Grund:", str(fehler)[:90])

Korrekt abgelehnt ✅
Grund: Abfrage abgelehnt (nur lesende MATCH...RETURN erlaubt): MATCH (n:Person) DETACH DELETE n


## 📝 Reflexion (abschließend fürs Graph-Modul)

1. Warum reicht die lesende Whitelist nicht mehr, sobald das System „aus Dokumenten lernen" (also schreiben) soll? (→ Freigabe in Lektion 15!)
2. Vergleiche: Was kann der Graph, was Vektor-Suche (Build 2) nicht kann – und umgekehrt? Schreib zwei Fragen als Beleg.

**Genau diese zwei Welten kombiniert Build 3** – nächste Lektion: LangGraph steuert den Ablauf.

## ➡️ Weiter

**🚀 Lektion 13:** `notebooks/07_langgraph/13_langgraph_workflows.ipynb` – LangGraph: State, Nodes, konditionale Kanten. Deine Steuerzentrale für Build 3.